# Amazon ML Challenge 2026 - Business Entity Resolution

Complete pipeline: Blocking → Features → Training → Prediction

**Data Path:** `/kaggle/input/datasets/summohith/amazon-ml-2026/student_resource/dataset/`

In [ ]:
# Install dependencies
!pip install -r /kaggle/working/MLSchool/requirements.txt -q

# Verify imports
import polars as pl
import xgboost as xgb
import rapidfuzz
import jellyfish
import sklearn
print("Polars:", pl.__version__)
print("XGBoost:", xgb.__version__)
print("RapidFuzz:", rapidfuzz.__version__)
print("Jellyfish:", jellyfish.__version__)
print("Scikit-learn:", sklearn.__version__)

In [ ]:
# Setup paths
import sys
sys.path.insert(0, '/kaggle/working/MLSchool/src')

from utils import BASE_PATH, OUTPUT_DIR
print("BASE_PATH:", BASE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

# Verify data exists
import os
for split in ['train', 'test']:
    for src in ['source1', 'source2', 'source3']:
        p = f"{BASE_PATH}/dataset/{split}/{split}_{src}.tsv"
        print(f"{p}: {'EXISTS' if os.path.exists(p) else 'MISSING'}")
print(f"GT: {'EXISTS' if os.path.exists(f'{BASE_PATH}/dataset/train/train_ground_truth.tsv') else 'MISSING'}")

In [ ]:
# ============================================================
# STAGE 1: BLOCKING / CANDIDATE GENERATION
# ============================================================
%run /kaggle/working/MLSchool/src/01_blocking.py

In [ ]:
# ============================================================
# STAGE 2: FEATURE EXTRACTION
# ============================================================
%run /kaggle/working/MLSchool/src/02_features.py

In [ ]:
# ============================================================
# STAGE 3: MODEL TRAINING
# ============================================================
%run /kaggle/working/MLSchool/src/03_model.py

In [ ]:
# ============================================================
# STAGE 4: PREDICTION & SUBMISSION
# ============================================================
%run /kaggle/working/MLSchool/src/04_predict.py

In [ ]:
# ============================================================
# VALIDATE SUBMISSION
# ============================================================
!python /kaggle/working/MLSchool/utils/validate_submission.py \
    --matching /kaggle/working/MLSchool/output/matching_results.tsv \
    --candidate /kaggle/working/MLSchool/output/candidate_pairs.tsv \
    --test-dir /kaggle/input/datasets/summohith/amazon-ml-2026/student_resource/dataset/test

In [ ]:
# Check output files
import polars as pl
matching = pl.read_csv('/kaggle/working/MLSchool/output/matching_results.tsv', separator='\t')
candidates = pl.read_csv('/kaggle/working/MLSchool/output/candidate_pairs.tsv', separator='\t')
print(f"Matching results: {len(matching):,} rows")
print(f"Candidate pairs: {len(candidates):,} rows")
print(f"Matching non-empty: {matching.filter(pl.col('matched_entity_ids') != '').height:,}")
print(f"Candidates non-empty: {candidates.filter(pl.col('candidate_entity_ids') != '').height:,}")

# Sample
print("\nMatching sample:")
print(matching.head(5))
print("\nCandidates sample:")
print(candidates.head(5))